In [1]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

In [2]:
BANK_NAME = "NH"
BANK_CODE = "NH"

NH_URL = "https://banking.nonghyup.com/servlet/IPEFP0662R.frag"
REFERER_URL = "https://banking.nonghyup.com/servlet/IPEFP0661I.view"

COMMON_HEADERS = {
    "accept": "text/html, */*; q=0.01",
    "content-type": "application/x-www-form-urlencoded; charset=UTF-8",
    "origin": "https://banking.nonghyup.com",
    "referer": REFERER_URL,
    "user-agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/146.0.0.0 Safari/537.36"
    ),
    "x-requested-with": "XMLHttpRequest",
}

In [3]:
def generate_month_end_dates(start_date="2004-01-31", end_date="2019-12-31"):
    dates = []
    current = pd.to_datetime(start_date) + pd.offsets.MonthEnd(0)
    end = pd.to_datetime(end_date)

    while current <= end:
        dates.append(current.strftime("%Y%m%d"))
        current = current + pd.offsets.MonthEnd(1)

    return dates

In [4]:
NH_TOKEN = "260330121427OEFIPINOPT0147721101"
NH_DEVICE_SESSION = "5251e362-f85d-45b6-9348-13034f8b91d7"

In [5]:
def fetch_nh_html(session, target_yyyymmdd, dp_ds, token, device_session, debug=False):
    """
    dp_ds
    - 01: 외화보통예금
    - 02: 외화정기예금
    """
    dt = pd.to_datetime(target_yyyymmdd, format="%Y%m%d")

    headers = COMMON_HEADERS.copy()
    headers["token"] = token

    payload = {
        "inq_bas_dt": target_yyyymmdd,
        "cmd": "1",
        "dp_ds": dp_ds,
        "start_year": dt.strftime("%Y"),
        "start_month": dt.strftime("%m"),
        "start_date": dt.strftime("%d"),
        "TOKEN": token,
        "DEVICE_SESSION": device_session,
    }

    resp = session.post(
        NH_URL,
        headers=headers,
        data=payload,
        timeout=60
    )
    resp.raise_for_status()

    if debug:
        print("status:", resp.status_code)
        print("content-type:", resp.headers.get("content-type"))
        print("response token:", resp.headers.get("token"))
        print(resp.text[:1000])

    return resp.text, resp.headers.get("token", token)

In [6]:
def parse_nh_demand_html(html, target_yyyymmdd):
    soup = BeautifulSoup(html, "html.parser")
    tables = soup.find_all("table")

    if len(tables) < 2:
        return pd.DataFrame(columns=[
            "bank", "bank_code", "target_date",
            "currency", "maturity", "rate", "product"
        ])

    rate_table = tables[1]
    tbody = rate_table.find("tbody")
    if tbody is None:
        return pd.DataFrame(columns=[
            "bank", "bank_code", "target_date",
            "currency", "maturity", "rate", "product"
        ])

    rows = []
    target_date_fmt = pd.to_datetime(target_yyyymmdd, format="%Y%m%d").strftime("%Y-%m-%d")

    for tr in tbody.find_all("tr"):
        cells = [c.get_text(" ", strip=True) for c in tr.find_all(["th", "td"])]

        # 기대 구조:
        # [국가1, 통화1, 이율1, 국가2, 통화2, 이율2]
        if len(cells) >= 3:
            country1 = cells[0]
            currency1 = cells[1]
            rate1 = pd.to_numeric(cells[2], errors="coerce")

            rows.append({
                "bank": BANK_NAME,
                "bank_code": BANK_CODE,
                "target_date": target_date_fmt,
                "currency": currency1,
                "maturity": "보통예금",
                "rate": rate1,
                "product": f"외화보통예금 ({country1})"
            })

        if len(cells) >= 6:
            country2 = cells[3]
            currency2 = cells[4]
            rate2 = pd.to_numeric(cells[5], errors="coerce")

            rows.append({
                "bank": BANK_NAME,
                "bank_code": BANK_CODE,
                "target_date": target_date_fmt,
                "currency": currency2,
                "maturity": "보통예금",
                "rate": rate2,
                "product": f"외화보통예금 ({country2})"
            })

    df = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
    return df

In [7]:
def parse_nh_time_html(html, target_yyyymmdd):
    soup = BeautifulSoup(html, "html.parser")
    tables = soup.find_all("table")

    if len(tables) < 2:
        return pd.DataFrame(columns=[
            "bank", "bank_code", "target_date",
            "currency", "maturity", "rate", "product"
        ])

    rate_table = tables[1]
    tbody = rate_table.find("tbody")
    if tbody is None:
        return pd.DataFrame(columns=[
            "bank", "bank_code", "target_date",
            "currency", "maturity", "rate", "product"
        ])

    maturity_cols = ["7일", "1개월", "2개월", "3개월", "6개월", "9개월", "12개월"]

    rows = []
    target_date_fmt = pd.to_datetime(target_yyyymmdd, format="%Y%m%d").strftime("%Y-%m-%d")
    current_currency = None

    for tr in tbody.find_all("tr"):
        cells = tr.find_all(["th", "td"])
        texts = [c.get_text(" ", strip=True) for c in cells]

        # 두 가지 패턴
        # 1) [통화, 종류, 7일, 1개월, ..., 12개월]
        # 2) [종류, 7일, 1개월, ..., 12개월]   <- rowspan 때문에 통화 생략
        if len(texts) == 9:
            current_currency = texts[0]
            resident_type = texts[1]
            rate_values = texts[2:]
        elif len(texts) == 8:
            resident_type = texts[0]
            rate_values = texts[1:]
        else:
            continue

        if current_currency is None:
            continue

        for maturity, rate_str in zip(maturity_cols, rate_values):
            rows.append({
                "bank": BANK_NAME,
                "bank_code": BANK_CODE,
                "target_date": target_date_fmt,
                "currency": current_currency,
                "maturity": maturity,
                "rate": pd.to_numeric(rate_str, errors="coerce"),
                "product": f"외화정기예금 ({resident_type})"
            })

    df = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
    return df

In [8]:
session = requests.Session()
test_date = "20260330"

# 외화보통예금
html_01, NH_TOKEN = fetch_nh_html(
    session=session,
    target_yyyymmdd=test_date,
    dp_ds="01",
    token=NH_TOKEN,
    device_session=NH_DEVICE_SESSION,
    debug=True
)
df_01 = parse_nh_demand_html(html_01, test_date)

print("외화보통예금 rows:", len(df_01))
print(df_01.head(10))

# 외화정기예금
html_02, NH_TOKEN = fetch_nh_html(
    session=session,
    target_yyyymmdd=test_date,
    dp_ds="02",
    token=NH_TOKEN,
    device_session=NH_DEVICE_SESSION,
    debug=True
)
df_02 = parse_nh_time_html(html_02, test_date)

print("외화정기예금 rows:", len(df_02))
print(df_02.head(14))

status: 200
content-type: text/html;charset=UTF-8
response token: 260330121427OEFIPINOPT0147721101



<script type="text/javascript">
	window["SERVICE_ID"] = "IPEFP0662R";
</script>

<script type="text/javascript">
	$.alopexready(function() {
		
			if(typeof TK_Rescan === "function") TK_Rescan();
			
	});
</script>

















<script type="text/javascript">
//<![CDATA[
//]]>
</script>	
			<div class="box_result">
				<h2 class="tit_type1">외화보통예금</h2>
				<table class="tb_row" summary="이 표는 기준일자, 현재일시 항목으로 구성되어 있습니다.">
					<caption>외화보통예금</caption>
					<colgroup>
						<col style="width:18%;">
						<col style="width:32%;">
						<col style="width:18%;">
						<col style="width:auto;">
					</colgroup>
					<tbody>
						<tr>
							<th class="t_center">기준일자</th>
							<td>2026/03/30</td>
							<th class="t_center">현재일시</th>
							<td>2026년 03월 30일 12시 27분 28초</td>
						</tr>
					</tbody>
				</table>

				<table summary="이 표는 나라별 국가, 통화, 이율 항목으로 구성되어 있습니다." class="tb_col t

In [9]:
df_test = pd.concat([df_01, df_02], ignore_index=True)
print("총 rows:", len(df_test))
print(df_test.head(20))

총 rows: 218
   bank bank_code target_date currency maturity  rate            product
0    NH        NH  2026-03-30      USD     보통예금  0.01        외화보통예금 (미국)
1    NH        NH  2026-03-30      JPY     보통예금  0.00        외화보통예금 (일본)
2    NH        NH  2026-03-30      EUR     보통예금  0.01      외화보통예금 (유럽연합)
3    NH        NH  2026-03-30      CNY     보통예금  0.00        외화보통예금 (중국)
4    NH        NH  2026-03-30      GBP     보통예금  0.01        외화보통예금 (영국)
5    NH        NH  2026-03-30      CHF     보통예금  0.00       외화보통예금 (스위스)
6    NH        NH  2026-03-30      CAD     보통예금  0.01       외화보통예금 (캐나다)
7    NH        NH  2026-03-30      HKD     보통예금  0.01        외화보통예금 (홍콩)
8    NH        NH  2026-03-30      SEK     보통예금  0.01       외화보통예금 (스웨덴)
9    NH        NH  2026-03-30      AUD     보통예금  0.01        외화보통예금 (호주)
10   NH        NH  2026-03-30      DKK     보통예금  0.01       외화보통예금 (덴마크)
11   NH        NH  2026-03-30      NOK     보통예금  0.01      외화보통예금 (노르웨이)
12   NH        NH  2026-03-30      AED 

In [10]:
def crawl_nh_range(
    start_date="2004-01-31",
    end_date="2019-12-31",
    token=NH_TOKEN,
    device_session=NH_DEVICE_SESSION,
    sleep_sec=0.3
):
    session = requests.Session()
    all_frames = []
    failed_dates = []

    date_list = generate_month_end_dates(start_date, end_date)
    print(f"총 {len(date_list)}개 날짜 수집 시작")

    current_token = token

    for i, yyyymmdd in enumerate(date_list, 1):
        try:
            # 1) 외화보통예금
            html_01, current_token = fetch_nh_html(
                session=session,
                target_yyyymmdd=yyyymmdd,
                dp_ds="01",
                token=current_token,
                device_session=device_session,
                debug=False
            )
            df_01 = parse_nh_demand_html(html_01, yyyymmdd)

            # 2) 외화정기예금
            html_02, current_token = fetch_nh_html(
                session=session,
                target_yyyymmdd=yyyymmdd,
                dp_ds="02",
                token=current_token,
                device_session=device_session,
                debug=False
            )
            df_02 = parse_nh_time_html(html_02, yyyymmdd)

            df_day = pd.concat([df_01, df_02], ignore_index=True)

            if not df_day.empty:
                all_frames.append(df_day)

            print(f"[{i}/{len(date_list)}] {yyyymmdd} 완료 - {len(df_day)} rows")

        except Exception as e:
            print(f"[{i}/{len(date_list)}] {yyyymmdd} 실패 - {e}")
            failed_dates.append({
                "target_date": yyyymmdd,
                "error": str(e)
            })

        time.sleep(sleep_sec)

    if all_frames:
        final_df = pd.concat(all_frames, ignore_index=True)
        final_df = final_df.drop_duplicates().reset_index(drop=True)
    else:
        final_df = pd.DataFrame(columns=[
            "bank", "bank_code", "target_date",
            "currency", "maturity", "rate", "product"
        ])

    failed_df = pd.DataFrame(failed_dates)
    return final_df, failed_df

In [11]:
nh_df, failed_df = crawl_nh_range(
    start_date="2004-01-31",
    end_date="2019-12-31",
    token=NH_TOKEN,
    device_session=NH_DEVICE_SESSION,
    sleep_sec=0.3
)

print("\n최종 shape:", nh_df.shape)
print(nh_df.head())
print("\n실패 건수:", len(failed_df))
print(failed_df.head())

총 192개 날짜 수집 시작
[1/192] 20040131 완료 - 196 rows
[2/192] 20040229 완료 - 196 rows
[3/192] 20040331 완료 - 196 rows
[4/192] 20040430 완료 - 196 rows
[5/192] 20040531 완료 - 196 rows
[6/192] 20040630 완료 - 196 rows
[7/192] 20040731 완료 - 196 rows
[8/192] 20040831 완료 - 196 rows
[9/192] 20040930 완료 - 196 rows
[10/192] 20041031 완료 - 196 rows
[11/192] 20041130 완료 - 196 rows
[12/192] 20041231 완료 - 196 rows
[13/192] 20050131 완료 - 196 rows
[14/192] 20050228 완료 - 196 rows
[15/192] 20050331 완료 - 196 rows
[16/192] 20050430 완료 - 196 rows
[17/192] 20050531 완료 - 196 rows
[18/192] 20050630 완료 - 196 rows
[19/192] 20050731 완료 - 196 rows
[20/192] 20050831 완료 - 196 rows
[21/192] 20050930 완료 - 196 rows
[22/192] 20051031 완료 - 196 rows
[23/192] 20051130 완료 - 196 rows
[24/192] 20051231 완료 - 196 rows
[25/192] 20060131 완료 - 196 rows
[26/192] 20060228 완료 - 196 rows
[27/192] 20060331 완료 - 196 rows
[28/192] 20060430 완료 - 196 rows
[29/192] 20060531 완료 - 196 rows
[30/192] 20060630 완료 - 196 rows
[31/192] 20060731 완료 - 196 rows
[

In [12]:
print("고유 날짜 수:", nh_df["target_date"].nunique())
print("\n날짜별 행 수 상위:")
print(nh_df["target_date"].value_counts().head())

print("\n상품별 행 수:")
print(nh_df["product"].value_counts().head(20))

print("\n샘플:")
print(nh_df.sample(min(20, len(nh_df)), random_state=42))

고유 날짜 수: 192

날짜별 행 수 상위:
target_date
2015-06-30    216
2015-07-31    216
2015-08-31    216
2015-09-30    216
2015-10-31    216
Name: count, dtype: int64

상품별 행 수:
product
외화정기예금 (거주자)       18816
외화정기예금 (비거주자)      18816
외화보통예금 (미국)           55
외화보통예금 (일본)           55
외화보통예금 (유럽연합)         55
외화보통예금 (중국)           55
외화보통예금 (영국)           55
외화보통예금 (스위스)          55
외화보통예금 (캐나다)          55
외화보통예금 (홍콩)           55
외화보통예금 (스웨덴)          55
외화보통예금 (호주)           55
외화보통예금 (덴마크)          55
외화보통예금 (노르웨이)         55
외화보통예금 (아랍에미리트)       55
외화보통예금 (싱가포르)         55
외화보통예금 (뉴질랜드)         55
외화보통예금 (태국)           55
외화보통예금 (인도네시아)        55
외화보통예금 (러시아)          55
Name: count, dtype: int64

샘플:
      bank bank_code target_date currency maturity  rate        product
35463   NH        NH  2018-09-30      NOK     12개월  1.49  외화정기예금 (비거주자)
21261   NH        NH  2013-01-31      CAD      2개월  0.99  외화정기예금 (비거주자)
23180   NH        NH  2013-11-30      CNY      3개월  4.58  외화정기예금 (비거주자)
3944    N

In [13]:
output_file = "nh_2004_2019_full.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    nh_df.to_excel(writer, sheet_name="Sheet1", index=False)
    failed_df.to_excel(writer, sheet_name="failed_dates", index=False)

print(f"저장 완료: {output_file}")

저장 완료: nh_2004_2019_full.xlsx
